# 02 – Baselines
Buy-and-hold, flat, and the previous linear model on the NEW data, with costs.
These are the bars every candidate must beat.

**What this notebook can and cannot say.** The previous linear model's parameters (windows, horizon 37, learning rate, L2, threshold) were tuned on Binance spot data up to 2026-08-01, which overlaps almost all of this series. Its results here are therefore **not out-of-sample with respect to those parameters**; they are a reference level, not evidence of an edge. Only the slice after 2026-08-01 was not available to that tuning, and it is short. No parameter is tuned in this notebook.

**Accounting** (tested in `tests/test_evaluation.py`): a position decided at the close of bar *t* is held over bar *t+1*; costs are charged on position changes; funding is charged per held hour. The sign and timing of funding are **assumed** (long pays a positive rate) and are shown with and without funding and with a one-bar lag.

Needs the cache written by notebook 01 (no network access).

Status: in progress

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlt.data.cache import FrameCache
from mlt.evaluation import metrics, simulate
from mlt.models.linear import OnlineLinear
from mlt.models.online import run_online
from mlt.protocol import DATA_END, DATA_START, block_mask, freeze

INSTRUMENT = "BTC_USDC-PERPETUAL"
INTERVAL = "1h"
cache = FrameCache("../data")
candles = freeze(cache.read("deribit", INSTRUMENT, "candles", INTERVAL))      # frozen window, ADR 0004
funding = freeze(cache.read("deribit", INSTRUMENT, "funding"))
print("frozen window:", DATA_START, "->", DATA_END)
close = candles["close"]
rate = funding["interest_1h"]
print(close.shape, close.index[0], "->", close.index[-1])
print("funding observations:", len(rate), "| candles without funding:", int(rate.reindex(close.index).isna().sum()))

## 1. Settings (inherited, not tuned here)
Windows, horizon, learning rate, L2 and threshold multiplier are the predecessor's values (`ml_trading`, linear notebook and app defaults). `BURN_IN` is the number of model updates before the first evaluated bar, so that the model is not judged while it is still untrained; it is an explicit choice, not a tuned one.

In [ ]:
WINDOWS = [125, 150, 175, 200, 225, 250]
HORIZON = 37
LR, L2 = 0.00325, 0.425
THRESHOLD_MULTIPLIER = 1.0     # threshold = multiplier x (fee + slippage), as in the app
ALLOW_SHORTS = True
CLIP = 0.2                     # the app clips predictions to +-0.2
BURN_IN = 2000                 # updates before the first evaluated bar
PREDECESSOR_END = pd.Timestamp("2026-08-01", tz="UTC")   # end of the data the predecessor tuned on

SCENARIOS = {
    "gateway taker 3.5 bp":          dict(fee=0.00035, slip=0.0000),
    "gateway taker + 5 bp slippage": dict(fee=0.00035, slip=0.0005),
    "predecessor costs (10 + 5 bp)": dict(fee=0.0010,  slip=0.0005),
}

## 2. Model predictions (causal, walk-forward)

In [ ]:
X = np.column_stack([np.log(close / close.shift(w)).to_numpy() for w in WINDOWS])
y = np.log(close.shift(-HORIZON) / close).to_numpy()
preds = pd.Series(
    np.clip(run_online(OnlineLinear(len(WINDOWS), LR, L2), X, y, HORIZON), -CLIP, CLIP),
    index=close.index,
)
first_update = max(WINDOWS) + HORIZON
start_i = first_update + BURN_IN
ev = close.index[start_i:]
print("first evaluated bar:", ev[0], "| bars evaluated:", len(ev))
print("prediction: mean %.5f  std %.5f  min %.4f  max %.4f" % (preds.loc[ev].mean(), preds.loc[ev].std(), preds.loc[ev].min(), preds.loc[ev].max()))

## 3. Positions and results
Baselines: **flat**, **buy-and-hold** (1x long) and the **linear model**, all on the same evaluated bars, 1x leverage (position = fraction of equity). Funding is included unless stated.

In [ ]:
def positions(pred, threshold):
    pos = pd.Series(0.0, index=pred.index)
    pos[pred > threshold] = 1.0
    if ALLOW_SHORTS:
        pos[pred < -threshold] = -1.0
    return pos

def run(name, target, fee, slip, use_funding=True, lag=0, window=ev):
    c = close.loc[window]
    f = rate if use_funding else None
    out = simulate(c, target.loc[window], fee, slip, funding_1h=f, funding_lag=lag)
    return out

def table(window=ev, use_funding=True, lag=0):
    rows = {}
    for sc, p in SCENARIOS.items():
        thr = abs(THRESHOLD_MULTIPLIER * (p["fee"] + p["slip"]))
        books = {
            "flat": pd.Series(0.0, index=close.index),
            "buy-and-hold": pd.Series(1.0, index=close.index),
            "linear model": positions(preds, thr),
        }
        for name, tgt in books.items():
            out = run(name, tgt, p["fee"], p["slip"], use_funding, lag, window)
            rows[(sc, name)] = metrics(out["net"], out["held"])
    return pd.DataFrame(rows).T

res = table()
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
res[["total_return", "cagr", "sharpe", "max_drawdown", "exposure", "trades", "turnover_per_year"]]

### Sensitivity to funding
Same table without funding and with funding applied one bar later. If a conclusion changes with these, the funding convention must be verified before it is trusted.

In [ ]:
cols = ["total_return", "sharpe", "max_drawdown"]
no_f = table(use_funding=False)[cols].add_suffix(" (no funding)")
lag1 = table(lag=1)[cols].add_suffix(" (lag 1)")
no_f.join(lag1)

## 4. Calendar years, and the slice the predecessor never saw

In [ ]:
rows = {}
sc = "gateway taker 3.5 bp"
p = SCENARIOS[sc]
thr = abs(THRESHOLD_MULTIPLIER * (p["fee"] + p["slip"]))
lin = positions(preds, thr)
hold = pd.Series(1.0, index=close.index)
slices = {str(y_): ev[ev.year == y_] for y_ in sorted(set(ev.year))}
slices["after 2026-08-01 (never tuned on)"] = ev[ev > PREDECESSOR_END]
for label, w in slices.items():
    if len(w) < 50:
        continue
    for name, tgt in {"buy-and-hold": hold, "linear model": lin}.items():
        o = run(name, tgt, p["fee"], p["slip"], True, 0, w)
        rows[(label, name)] = metrics(o["net"], o["held"])
pd.DataFrame(rows).T[["bars", "total_return", "sharpe", "max_drawdown", "trades"]]

### Per-block results (ADR 0004)
Baselines on the **development** and **validation** blocks. The **test** block is deliberately not evaluated here: seeing how the final period went, even for a baseline, would let us tune toward it. Its baseline numbers are produced in notebook 04 together with the final candidate. The model is evaluated from the first bar after warm-up and burn-in, so the development block starts later than 2023-10-29.

In [ ]:
rows = {}
for blk in ("development", "validation"):
    w = ev[block_mask(ev, blk)]
    for name, tgt in {"buy-and-hold": hold, "linear model": lin}.items():
        o = run(name, tgt, p["fee"], p["slip"], True, 0, w)
        rows[(blk, name)] = metrics(o["net"], o["held"])
pd.DataFrame(rows).T[["bars", "total_return", "sharpe", "max_drawdown", "trades"]]

## 5. Is the model's timing better than luck?
The linear model's position series is shifted circularly by random amounts (at least 1,000 bars). Exposure, number of trades and costs stay exactly the same; only the alignment with the market is destroyed. If the real Sharpe is not clearly above this distribution, the timing carries no evidence.

In [ ]:
rng = np.random.default_rng(0)
base = run("linear model", lin, p["fee"], p["slip"])
real = metrics(base["net"], base["held"])["sharpe"]
pos = lin.loc[ev].to_numpy()
shifted = []
for k in rng.integers(1000, len(pos) - 1000, size=200):
    t = pd.Series(np.roll(pos, k), index=ev)
    o = simulate(close.loc[ev], t, p["fee"], p["slip"], funding_1h=rate)
    shifted.append(metrics(o["net"])["sharpe"])
shifted = np.array(shifted)
print("real Sharpe: %.3f | shifted: mean %.3f, 5th-95th pct [%.3f, %.3f]" % (real, shifted.mean(), *np.percentile(shifted, [5, 95])))
print("share of shifted runs with a Sharpe at least as high as the real one: %.1f %%" % (100 * (shifted >= real).mean()))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for name, tgt in {"buy-and-hold": hold, "linear model": lin, "flat": pd.Series(0.0, index=close.index)}.items():
    o = run(name, tgt, p["fee"], p["slip"])
    ax.plot(o["equity"], label=name)
ax.set_yscale("log"); ax.set_title(f"Equity, {sc}, funding included"); ax.legend(); plt.tight_layout()

## Findings (owner run, data to 2026-10-29 08:00 UTC; only numbers printed above)
- Evaluated period and number of bars: 2024-02-02 06:00 -> 2026-10-29 08:00 UTC, 24,003 hourly bars (after warm-up and a burn-in of 2,000 updates); the cache held 26,290 candles from 2023-10-29 23:00 (re-fetched after 01_data, so slightly different from R-001).
- Buy-and-hold, gateway taker 3.5 bp, funding included: total return +57.9 %, Sharpe 0.59, max drawdown -54.6 %.
- Linear model, same scenario: +92.1 %, Sharpe 0.745, max drawdown -35.3 %, 165 trades, in the market 96 % of the time. With +5 bp slippage: +78.9 % (Sharpe 0.695); with the predecessor's costs (10 + 5 bp): +16.6 % (Sharpe 0.347).
- Funding: buy-and-hold earns +85.1 % without funding and +57.9 % with it (a drag of about 27 percentage points); the one-bar lag changes nothing visible (+57.86 % vs +57.88 %). The sign and timing of funding are still assumed.
- Calendar years (gateway taker, funding): 2024 buy-and-hold +96.2 % vs linear +61.9 %; 2025 -9.8 % vs +3.0 %; 2026 to date -11.1 % vs +15.4 %; after 2026-08-01 (never tuned on, 2,144 bars) buy-and-hold +25.3 % (Sharpe 2.82) vs linear +10.4 % (Sharpe 1.36).
- Per-block results (development and validation; the test block is not evaluated here): to be filled after the re-run.
- Timing test: real Sharpe 0.745 vs shifted runs with mean -0.086 and 5th-95th percentile [-1.053, 0.728]; 5.0 % of the shifted runs reach at least the real value.
- Conclusion: the bar to beat at gateway costs with funding is Sharpe 0.745 / +92.1 % / drawdown -35.3 % (linear) and Sharpe 0.59 / +57.9 % / -54.6 % (buy-and-hold), measured on parameters that were tuned on this period, so it is a reference level, not an edge. The linear model is far from established as better: its advantage over buy-and-hold is small relative to the noise of a 2.75-year sample, it is very sensitive to costs, it trails buy-and-hold in the only slice it was not tuned on, and the timing test is borderline. Still open: the 13 zero-volume candles, the 36 candles without funding (treated as zero funding, counted), the funding convention, the real fee schedule, and the held-out split (ADR 0004).

Recorded as `R-002` in `docs/RESEARCH_LOG.md`.